In [1]:
#!pip uninstall xgboost

In [2]:
!pip install optuna xgboost==3.1.1 lightgbm fastparquet imbalanced-learn torch


In [3]:
import numpy as np
import pandas as pd
import os, joblib,time


from sklearn.model_selection import StratifiedKFold, cross_val_score,train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.calibration import CalibratedClassifierCV

import optuna
from optuna.pruners import SuccessiveHalvingPruner
from optuna.exceptions import TrialPruned

#from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer


from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score,
    classification_report, brier_score_loss, log_loss,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, RocCurveDisplay
)

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

import matplotlib.pyplot as plt
from functools import partial

import subprocess
import xgboost as xgb
import lightgbm as lgb
import torch

import warnings
warnings.filterwarnings('ignore')

### Carregando base de dados

In [4]:
df =pd.read_parquet('df_hepat_processed_v1.parquet',engine='fastparquet')
df = df.sample(frac=0.20, random_state=42)



### Selecionar variáveis relevantes para triagem


In [5]:

triage_features = [
    'NU_IDADE_N','CS_SEXO','CS_GESTANT','CS_RACA','CS_ESCOL_N',
    'ID_OCUPA_N','CON_AREA','CON_AMBIEN',
    'ANT_CB_LAM','ANT_CB_CRI','ANT_CB_SIN','ANT_CB_COR','ANT_CB_ROE',
    'ANT_CB_TER','ANT_CB_LIX','ANT_CB_OUT','ANT_ANIMAI','ANT_HUMANO',
    'CLI_FEBRE','CLI_MIALGI','CLI_CEFALE','CLI_PROST','CLI_CONGES',
    'CLI_PANTUR','CLI_VOMITO','CLI_DIARRE','CLI_ICTERI','CLI_RENAL',
    'CLI_RESPIR','CLI_CARDIA','CLI_HEMOPU','CLI_HEMORR','CLI_MENING',
    'CLI_OUTROS','SG_UF', 'baixa_escolaridade','raca_vulneravel','faixa_vulneravel',
    'vulnerabilidade_social', 'nivel_vulnerabilidade','mes_sintomas','estacao', 'periodo_chuvoso'
]



target_col = 'CLASSI_FIN'  # rótulo: confirmado vs descartado

if target_col == 'CLASSI_FIN':
    df = df[df['CLASSI_FIN'].isin(['Confirmado', 'Descartado'])].copy()

    # Criar variável alvo binária
    df['alvo'] = df['CLASSI_FIN'].map({
        'Confirmado': 1,
        'Descartado': 0
    })

X_data=df[triage_features]
y_data = df['alvo']


## Train framework

### Funções de apoio

In [6]:


def detect_gpu():
    """
    Detecta corretamente suporte a GPU para CUDA, XGBoost e LightGBM.
    Retorna um dicionário completo.
    """

    # 1. Verificar CUDA geral
    try:
        cuda_available = torch.cuda.is_available()
    except:
        cuda_available = False


    # 2. Testar LightGBM GPU

    try:
        # Testa se LightGBM foi compilado com suporte a GPU
        lgb_gpu = lgb.register_logger(lambda x: x)  # dummy
        booster = lgb.LGBMClassifier(device="gpu")
        lgb_gpu = True
    except:
        lgb_gpu = False


    # 3. Detalhes da GPU via nvidia-smi

    try:
        smi_output = subprocess.check_output("nvidia-smi", shell=True).decode()
        gpu_info = smi_output.split("\n")[2].strip()
    except:
        gpu_info = "Nenhuma GPU detectada (nvidia-smi não disponível)"

    return {
        "cuda_available": cuda_available,
        "lightgbm_gpu": lgb_gpu,
        "gpu_info": gpu_info
    }


In [7]:
def get_model_params_gpu(model_name, gpu_info):
    """
    Retorna hiperparâmetros base apropriados para CPU ou GPU.
    """
    use_gpu = gpu_info["cuda_available"]

    if model_name == "xgboost":
        if use_gpu:
            return {
                "tree_method": "gpu_hist",
                "predictor": "gpu_predictor"
            }
        else:
            return {
                "tree_method": "hist",
                "predictor": "cpu_predictor"
            }

    elif model_name == "lightgbm":
        if use_gpu:
            return {
                "device": "gpu",
                "gpu_platform_id": 0,
                "gpu_device_id": 0
            }
        else:
            return {
                "device": "cpu"
            }

    elif model_name in ["random_forest", "svm"]:
        # sklearn não usa GPU
        return {}

    return {}


In [8]:
def get_feature_types(df):
    """
    Identifica automaticamente colunas categóricas e numéricas de um dataframe.

    Retorna:
        cat_cols  -> lista de colunas categóricas
        num_cols  -> lista de colunas numéricas
    """

    # CATEGÓRICAS: object, category ou boolean
    cat_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    # NUMÉRICAS: int, float
    num_cols = df.select_dtypes(include=['int', 'float', 'int64', 'float64']).columns.tolist()

    return cat_cols, num_cols


###  1. FUNÇÃO PARA CRIAR O MODELO A PARTIR DO TRIAL (Optuna)

In [9]:
def create_model(trial, gpu_info):
    model_name = trial.suggest_categorical(
        "model_name", ["xgboost", "lightgbm", "random_forest", "svm"]
    )

    gpu_params = get_model_params_gpu(model_name, gpu_info)

    # RANDOM FOREST CPU
    if model_name == "random_forest":
        return RandomForestClassifier(
            n_estimators=trial.suggest_int("n_estimators", 5, 30),
            max_depth=trial.suggest_int("max_depth", 3, 9),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
            random_state=42,
            **gpu_params
        )

    # SVM CPU (sklearn não tem GPU)
    elif model_name == "svm":
        return SVC(
            kernel=trial.suggest_categorical("kernel", ["linear", "rbf"]),
            C = trial.suggest_float("C", 0.1, 3.0),
            gamma = trial.suggest_float("gamma", 0.001, 0.1),
            probability=True,
            **gpu_params
        )

    # XGBOOST GPU/CPU
    elif model_name == "xgboost":
        return XGBClassifier(
            n_estimators=trial.suggest_int("n_estimators", 5, 30),
            max_depth=trial.suggest_int("max_depth", 3, 9),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
            #**gpu_params
        )

    # LIGHTGBM GPU/CPU
    elif model_name == "lightgbm":
        return LGBMClassifier(
            n_estimators=trial.suggest_int("n_estimators", 5, 30),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3),
            max_depth=trial.suggest_int("max_depth", 3, 9),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            random_state=42,
           force_col_wise=True,
            **gpu_params
        )


### 2. FUNÇÃO OBJETIVO DO OPTUNA (COM CROSS-VALIDATION)


In [10]:
def build_pipeline(model, cat_cols, num_cols):

    # construindo o preprocessor
    num_preprocessor = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    cat_preprocessor = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown='infrequent_if_exist'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_preprocessor, num_cols),
            ("cat", cat_preprocessor, cat_cols)
        ]
    )

    # construindo pipe
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("model", model)
    ])

    return pipe


In [11]:
def objective(trial, X_train, y_train, cat_cols, num_cols,feature_names, gpu_info):

    model = create_model(trial, gpu_info)

    pipe = build_pipeline(model ,cat_cols, num_cols)


    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = []
    for step, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):

        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]


        pipe.fit(X_tr, y_tr)
        fold_auc = roc_auc_score(y_val, pipe.predict_proba(X_val)[:, 1])
        scores.append(fold_auc)

        #  reporta o valor parcial ao Optuna
        trial.report(fold_auc, step=step)

        #  aborta a tentativa dado a avaliação do pruner
        if trial.should_prune():
            raise TrialPruned()


    return np.mean(scores)


In [12]:
# salva os logs das tentativas com métricas, tempo de treinamento, e parametros utilizados pelos modelos
def dump_trials(study, trial):
    df_metrics = study.trials_dataframe(attrs=("number", "value", "params", "state","duration"))
    df_metrics.to_csv("log_opt/trials_log_v2.csv", index=False)


### 3. VALIDAÇÃO EXTERNA (OUTER CV) + OPTUNA INTERNO


In [13]:
def outer_cv_training_with_results(
    X, y, cat_cols, num_cols, n_trials=30, n_splits=5
):

    gpu_info = detect_gpu()
    print("\n🔍 GPU detectada:", gpu_info)

    outer_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = []
    feature_names = X.columns
    fold = 1
    for train_idx, test_idx in outer_cv.split(X, y):

        print(f"\n==============================")
        print(f"  Fold Externo {fold}/{n_splits}")
        print(f"==============================")

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pruner = SuccessiveHalvingPruner(min_resource=1,
                                 reduction_factor=3,
                                 min_early_stopping_rate=1)

        study = optuna.create_study(direction="maximize")

        objective_fixed = partial(objective,
                          X_train=X_train,
                          y_train=y_train,
                          cat_cols=cat_cols,
                          num_cols=num_cols,
                          feature_names=feature_names,
                          gpu_info=gpu_info)

        study.optimize(objective_fixed,
                      n_trials=n_trials,
                      callbacks=[dump_trials],
                      timeout=1800,
                      n_jobs=-1,
                      )

        best_params = study.best_params
        best_model_name = best_params["model_name"]
        best_model = create_model(study.best_trial, gpu_info)


        pipe = build_pipeline(best_model ,cat_cols, num_cols)

        # treino com timer
        start_time = time.time()
        pipe.fit(X_train, y_train)
        train_time = time.time() - start_time

        # CHECKPOINT
        checkpoint_dir = "checkpoints_best_models"
        if not os.path.exists(checkpoint_dir):
            os.makedirs(checkpoint_dir)

        checkpoint_path = os.path.join(
            checkpoint_dir, f"best_model_fold_{fold}.pkl"
        )
        joblib.dump(pipe, checkpoint_path)

        # predição
        y_pred = pipe.predict(X_test)
        y_pred_prob = pipe.predict_proba(X_test)[:, 1]

        # métricas
        auc = roc_auc_score(y_test, y_pred_prob)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)

        # salvar dados do fold
        results.append({
            "fold": fold,
            "best_model": best_model_name,
            "auc": auc,
            "accuracy": acc,
            "f1": f1,
            "recall": rec,
            "precision": prec,
            "hyperparameters": best_params,
            "checkpoint_path": checkpoint_path,
            "train_time_seconds": train_time,
            #"test_data": {
            #    "X_test": X_test.copy(),
            #    "y_test": y_test.reset_index(drop=True),
            #    "y_pred": pd.Series(y_pred)
            #}
        })

        fold += 1

    return results


# Execute Framework

In [14]:
# ================================================
# CRIA DIRETÓRIO DE CHECKPOINTS (se não existir)
# ================================================
import os, joblib
checkpoint_dir = "checkpoints_best_models"

if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)

checkpoint_dir = "log_opt"
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)


# ================================================


In [ ]:
cat_cols, num_cols = get_feature_types(X_data)
df_results = outer_cv_training_with_results(
    X_data, y_data,
    cat_cols=cat_cols,
    num_cols=num_cols,
    n_trials=30,
    n_splits=5
)


[I 2025-11-16 22:08:47,866] A new study created in memory with name: no-name-c7aa0121-7169-46e1-8925-500d331ffd3a



🔍 GPU detectada: {'cuda_available': False, 'xgboost_gpu': True, 'lightgbm_gpu': False, 'gpu_info': 'Nenhuma GPU detectada (nvidia-smi não disponível)'}

  Fold Externo 1/5


[I 2025-11-16 22:09:32,389] Trial 1 finished with value: 0.9233608988648518 and parameters: {'model_name': 'lightgbm', 'n_estimators': 10, 'learning_rate': 0.2806017124125312, 'max_depth': 5, 'subsample': 0.7844853483071592, 'colsample_bytree': 0.9724345933613345}. Best is trial 1 with value: 0.9233608988648518.
[I 2025-11-16 22:10:09,860] Trial 2 finished with value: 0.9095930551308715 and parameters: {'model_name': 'random_forest', 'n_estimators': 24, 'max_depth': 4, 'min_samples_split': 5, 'min_samples_leaf': 1}. Best is trial 1 with value: 0.9233608988648518.


#### Conferir checkpoints intermediarios

In [ ]:
df_results.to_parquet('df_results.parquet')

In [ ]:
df_metrics = pd.read_csv('log_opt/trials_log_v2.csv')
display(df_metrics)
print("Melhor configuração:", study.best_params)
print("Melhor score de ROC AUC:", study.best_value)

Carregando o pkl

In [ ]:
import joblib

modelo = joblib.load("checkpoints_best_models/best_model_fold_1.pkl")


In [ ]:
modelo

In [ ]:
modelo.named_steps["preprocessor"].get_feature_names_out()

In [ ]:
modelo.predict(X_data.iloc[-10:])